# Segmentation alimentaire avec FoodSeg103 et YOLOv11-seg

## Objectif

Ce notebook entraîne un modèle de segmentation alimentaire à partir du dataset FoodSeg103.

L’objectif est de segmenter les zones alimentaires dans une image afin d’estimer la proportion occupée par les aliments dans un plat.

## Dataset

Source : FoodSeg103 sur Kaggle  
Chemin Kaggle : `/kaggle/input/datasets/fontainenathan/foodseg103`

Le dataset contient :
- des images dans `Images/img_dir`
- des masques de segmentation dans `Images/ann_dir`
- les listes d’entraînement et de test dans `ImageSets`

## Sorties

Le notebook génère :
- un dataset converti au format YOLO segmentation
- un modèle entraîné `best.pt`
- des résultats de validation

In [36]:
from pathlib import Path

# Le notebook est dans notebooks/train_local/
PROJECT_ROOT = Path.cwd()

# Remonte dans l'arborescence jusqu'à trouver .dvc
while not (PROJECT_ROOT / ".dvc").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Impossible de trouver la racine du projet (dossier contenant .dvc).")
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Projet : {PROJECT_ROOT}")

Projet : c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation


In [ ]:
# ==========================================================
# Configuration de la source des données
# ==========================================================

USE_DVC = True


# Limite souhaitée
LIMIT_TRAIN = 5000

import os
import subprocess
import sys
import importlib.metadata

import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics", "mlflow", "python-dotenv"],
    check=True
)

In [ ]:
# Import des librairies nécessaires.
import os
import cv2
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from IPython.display import FileLink
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# ==========================================================
# Se placer automatiquement à la racine du projet
# ==========================================================


while not (PROJECT_ROOT / ".dvc").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Impossible de trouver la racine du projet.")
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

print(f"Projet : {PROJECT_ROOT}")

# ==========================================================
# Installation DVC
# ==========================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "dvc",
        "dvc-azure",
    ],
    check=True,
)

# ==========================================================
# Configuration Azure
# ==========================================================

CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

assert CONNECTION_STRING, (
    "AZURE_STORAGE_CONNECTION_STRING introuvable."
)

subprocess.run(
    [
        "dvc",
        "remote",
        "modify",
        "azure",
        "connection_string",
        CONNECTION_STRING,
    ],
    check=True,
)

print("\nDVC Remote :")
subprocess.run(["dvc", "remote", "list"], check=True)

print("\nTéléchargement du dataset...")
subprocess.run(["dvc", "pull"], check=True)

print("\nDataset téléchargé avec succès.")

Projet : c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation

DVC Remote :

Téléchargement du dataset...


CalledProcessError: Command '['dvc', 'pull']' returned non-zero exit status 1.

In [ ]:
from pathlib import Path

ROOT = PROJECT_ROOT / "dataset"

IMG_DIR = ROOT / "Images" / "img_dir"
ANN_DIR = ROOT / "Images" / "ann_dir"

CATEGORY_FILE = ROOT / "category_id.txt"

TRAIN_LIST = ROOT / "ImageSets" / "train.txt"
TEST_LIST = ROOT / "ImageSets" / "test.txt"

DEST = PROJECT_ROOT / "dataset"

print(ROOT)

c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\dataset


In [ ]:
import random

# Lecture des fichiers train.txt et test.txt.
def lire_liste(path):
    with open(path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_files = lire_liste(TRAIN_LIST)
val_files = lire_liste(TEST_LIST)

# Sauvegarde des tailles originales
original_train = len(train_files)
original_val = len(val_files)

# Calcul du ratio original val/train
val_ratio = original_val / original_train

# Nombre proportionnel de validation
if LIMIT_TRAIN is not None:
    LIMIT_VAL = int(LIMIT_TRAIN * val_ratio)
else:
    LIMIT_VAL = None

# Randomisation pour éviter le biais
import random

# Seed aléatoire à chaque exécution
SEED = random.randint(1, 999999)

print("Seed utilisée :", SEED)

random.shuffle(train_files)
random.shuffle(val_files)

# Réduction du dataset
if LIMIT_TRAIN is not None:
    train_files = train_files[:LIMIT_TRAIN]
    val_files = val_files[:LIMIT_VAL]

print("Train original:", original_train)
print("Val original:", original_val)
print("Nouveau train:", len(train_files))
print("Nouveau val:", len(val_files))
print("Ratio conservé:", round(len(val_files) / len(train_files), 4))

print("Exemples train:", train_files[:5])
print("Exemples val:", val_files[:5])

Seed utilisée : 750339
Train original: 4983
Val original: 2135
Nouveau train: 42
Nouveau val: 17
Ratio conservé: 0.4048
Exemples train: ['00003102.jpg', '00003359.jpg', '00000069.jpg', '00004007.jpg', '00003688.jpg']
Exemples val: ['00004573.jpg', '00005067.jpg', '00006889.jpg', '00005563.jpg', '00005797.jpg']


In [ ]:
# Lecture des catégories FoodSeg103.
# Le masque utilise : 0 = background, 1 à 103 = classes alimentaires.
# YOLO doit utiliser : 0 à 102 = classes alimentaires, sans background.

categories_foodseg = {}

with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)  # sépare l'ID et le nom, même si le nom contient des espaces
        if len(parts) == 2:
            id_foodseg = int(parts[0])
            nom_classe = parts[1].strip()
            categories_foodseg[id_foodseg] = nom_classe

# On ignore le background et on décale les IDs pour YOLO.
categories_yolo = {
    id_foodseg - 1: nom
    for id_foodseg, nom in categories_foodseg.items()
    if id_foodseg != 0
}

print("Nombre de classes YOLO:", len(categories_yolo))
print("Exemples:")
for i in list(categories_yolo.keys())[:10]:
    print(i, "->", categories_yolo[i])

# Exemple important : FoodSeg 66 = rice, donc YOLO 65 = rice.
print("FoodSeg 48 -> YOLO", 48 - 1, "=", categories_foodseg.get(48))
print("FoodSeg 66 -> YOLO", 66 - 1, "=", categories_foodseg.get(66))
print("FoodSeg 90 -> YOLO", 90 - 1, "=", categories_foodseg.get(90))


Nombre de classes YOLO: 103
Exemples:
0 -> candy
1 -> egg tart
2 -> french fries
3 -> chocolate
4 -> biscuit
5 -> popcorn
6 -> pudding
7 -> ice cream
8 -> cheese butter
9 -> cake
FoodSeg 48 -> YOLO 47 = chicken duck
FoodSeg 66 -> YOLO 65 = rice
FoodSeg 90 -> YOLO 89 = snow peas


In [ ]:
# Création de la structure attendue par YOLO.
for d in ["images/train", "images/val", "labels/train", "labels/val"]:
    (DEST / d).mkdir(parents=True, exist_ok=True)

print("Structure YOLO créée.")

Structure YOLO créée.


In [ ]:
# Conversion d'un masque PNG en fichier label YOLO segmentation multi-classes.
# Important : on conserve les vraies catégories du masque FoodSeg103.
# 0 = background, ignoré.
# 1 à 103 = classes alimentaires FoodSeg103.
# YOLO attend des classes de 0 à 102, donc : classe_yolo = valeur_masque - 1.

def mask_to_yolo_multiclass(mask_path, output_txt, min_area=20):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return False

    if mask.ndim == 3:
        h, w = mask.shape[:2]
        mask = mask[:, :, 0]
    else:
        h, w = mask.shape
    lignes = []

    valeurs_classes = np.unique(mask)

    for valeur in valeurs_classes:
        if valeur == 0:
            continue  # background

        classe_yolo = int(valeur) - 1

        # Sécurité : ignorer une classe inconnue.
        if classe_yolo not in categories_yolo:
            continue

        masque_classe = (mask == valeur).astype(np.uint8) * 255

        contours, _ = cv2.findContours(
            masque_classe,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < 3:
                continue

            if cv2.contourArea(contour) < min_area:
                continue

            # Simplification légère pour éviter des polygones trop longs.
            epsilon = 0.001 * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, epsilon, True)

            if len(contour) < 3:
                continue

            points = contour.reshape(-1, 2)
            coords = []

            for x, y in points:
                coords.append(x / w)
                coords.append(y / h)

            ligne = str(classe_yolo) + " " + " ".join([f"{c:.6f}" for c in coords])
            lignes.append(ligne)

    with open(str(output_txt), "w", encoding="utf-8") as f:
        f.write("\n".join(lignes))

    return len(lignes) > 0


In [ ]:
# Prendre un masque de test

mask_path = ANN_DIR / "train" / f"{Path(train_files[0]).stem}.png"

print(mask_path)

mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

print(type(mask))
print(mask.shape)
print(mask.dtype)

c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\dataset\Images\ann_dir\train\00003102.png
<class 'numpy.ndarray'>
(1366, 2048, 1)
uint8


In [ ]:
# Fonction pour trouver le masque correspondant à une image.
# Exemple : 00000001.jpg -> 00000001.png

def trouver_masque(nom_image, split):
    base = Path(nom_image).stem
    mask_path = ANN_DIR / split / f"{base}.png"

    if mask_path.exists():
        return mask_path

    return None


# Conversion d'un split train ou test vers la structure YOLO.
def convertir_split(files, src_split, dst_split):
    ok = 0
    manquants = 0
    sans_masque_valide = 0

    for nom_image in files:
        img_path = IMG_DIR / src_split / nom_image
        mask_path = trouver_masque(nom_image, src_split)

        if not img_path.exists() or mask_path is None:
            manquants += 1
            continue

        base = Path(nom_image).stem

        dst_img = DEST / "images" / dst_split / nom_image
        dst_label = DEST / "labels" / dst_split / f"{base}.txt"

        shutil.copy2(str(img_path), str(dst_img))

        if mask_to_yolo_multiclass(mask_path, dst_label):
            ok += 1
        else:
            sans_masque_valide += 1

    print(f"{dst_split} ok :", ok)
    print(f"{dst_split} manquants :", manquants)
    print(f"{dst_split} sans masque valide :", sans_masque_valide)


convertir_split(train_files, "train", "train")
convertir_split(val_files, "test", "val")

train ok : 42
train manquants : 0
train sans masque valide : 0
val ok : 17
val manquants : 0
val sans masque valide : 0


In [ ]:
# Vérification finale de la conversion.

train_images = DEST / "images" / "train"
train_labels = DEST / "labels" / "train"
val_images = DEST / "images" / "val"
val_labels = DEST / "labels" / "val"

print("Images train :", len(list(train_images.iterdir())))
print("Labels train :", len(list(train_labels.iterdir())))
print("Images val :", len(list(val_images.iterdir())))
print("Labels val :", len(list(val_labels.iterdir())))

print("Exemples labels :")
print(sorted([f.name for f in train_labels.iterdir()])[:5])

Images train : 42
Labels train : 42
Images val : 17
Labels val : 17
Exemples labels :
['00000069.txt', '00000142.txt', '00000180.txt', '00000199.txt', '00000352.txt']


In [ ]:
# Vérification du contenu d'un fichier label.

train_labels = DEST / "labels" / "train"

exemple_label = sorted(train_labels.iterdir())[0]

print(f"Fichier : {exemple_label.name}")

with open(exemple_label, "r", encoding="utf-8") as f:
    print(f.read()[:500])

Fichier : 00000069.txt
29 0.554688 0.002604 0.552734 0.049479 0.562500 0.075521 0.539062 0.096354 0.558594 0.098958 0.611328 0.117188 0.646484 0.145833 0.673828 0.192708 0.671875 0.210938 0.679688 0.213542 0.687500 0.236979 0.722656 0.257812 0.730469 0.278646 0.744141 0.278646 0.833984 0.375000 0.875000 0.424479 0.884766 0.419271 0.912109 0.393229 0.927734 0.369792 0.937500 0.343750 0.941406 0.302083 0.939453 0.281250 0.927734 0.252604 0.929688 0.247396 0.935547 0.247396 0.951172 0.268229 0.960938 0.276042 0.974609 0.


In [ ]:
# Création du fichier de configuration YOLO multi-classes.
# On utilise les 103 classes alimentaires de FoodSeg103, sans le background.

from pathlib import Path

DATASET_YAML = PROJECT_ROOT / "dataset.yaml"

yaml_content = f"""
path: {DEST.as_posix()}
train: images/train
val: images/val

names:
"""

for idx in sorted(categories_yolo.keys()):
    nom = categories_yolo[idx].replace("'", "")
    yaml_content += f"  {idx}: '{nom}'\n"

with open(DATASET_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"dataset.yaml créé : {DATASET_YAML.resolve()}\n")
print(yaml_content[:1500])

dataset.yaml créé : C:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\dataset.yaml


path: c:/Users/cleri/OneDrive - Collège de Bois-de-Boulogne/curso_ai/PROJET DE SYNTHÈSE/Food Segmentation/dataset
train: images/train
val: images/val

names:
  0: 'candy'
  1: 'egg tart'
  2: 'french fries'
  3: 'chocolate'
  4: 'biscuit'
  5: 'popcorn'
  6: 'pudding'
  7: 'ice cream'
  8: 'cheese butter'
  9: 'cake'
  10: 'wine'
  11: 'milkshake'
  12: 'coffee'
  13: 'juice'
  14: 'milk'
  15: 'tea'
  16: 'almond'
  17: 'red beans'
  18: 'cashew'
  19: 'dried cranberries'
  20: 'soy'
  21: 'walnut'
  22: 'peanut'
  23: 'egg'
  24: 'apple'
  25: 'date'
  26: 'apricot'
  27: 'avocado'
  28: 'banana'
  29: 'strawberry'
  30: 'cherry'
  31: 'blueberry'
  32: 'raspberry'
  33: 'mango'
  34: 'olives'
  35: 'peach'
  36: 'lemon'
  37: 'pear'
  38: 'fig'
  39: 'pineapple'
  40: 'grape'
  41: 'kiwi'
  42: 'melon'
  43: 'orange'
  44: 'watermelon'
  45: 'steak'


In [ ]:
import os
from pathlib import Path

import mlflow

MLFLOW_USER = os.getenv("MLFLOW_USER")
MLFLOW_PASS = os.getenv("MLFLOW_PASS")
MLFLOW_HOST = os.getenv("MLFLOW_HOST")

assert MLFLOW_USER, "MLFLOW_USER introuvable"
assert MLFLOW_PASS, "MLFLOW_PASS introuvable"
assert MLFLOW_HOST, "MLFLOW_HOST introuvable"

TRACKING_URI = (
    f"http://{MLFLOW_USER}:{MLFLOW_PASS}@{MLFLOW_HOST}:5000"
)

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("foodseg-yolo")

# ==========================================================
# Paramètres
# ==========================================================

MODEL_NAME = "yolo11s-seg.pt"
EPOCHS = 150
IMGSZ = 768
BATCH = 16

# ==========================================================
# Training + Logging
# ==========================================================

with mlflow.start_run() as run:

    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("imgsz", IMGSZ)
    mlflow.log_param("batch", BATCH)
    mlflow.log_param("seed", SEED)
    mlflow.log_param("limit_train", LIMIT_TRAIN)

    mlflow.set_tag("task", "segmentation")
    mlflow.set_tag("dataset", "foodseg103")

    model = YOLO(MODEL_NAME)

    train_results = model.train(
        data="dataset.yaml",
        task="segment",
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
    )

    metrics = model.val()

    mlflow.log_metric("map50_box", metrics.box.map50)
    mlflow.log_metric("map75_box", metrics.box.map75)
    mlflow.log_metric("map_box", metrics.box.map)

    mlflow.log_metric("map50_mask", metrics.seg.map50)
    mlflow.log_metric("map75_mask", metrics.seg.map75)
    mlflow.log_metric("map_mask", metrics.seg.map)

    run_dir = Path(train_results.save_dir)

    mlflow.log_artifact(str(run_dir / "results.csv"))
    mlflow.log_artifact(str(run_dir / "results.png"))
    mlflow.log_artifact(str(run_dir / "confusion_matrix.png"))
    mlflow.log_artifact(str(run_dir / "weights" / "best.pt"))

c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/08 11:06:38 INFO mlflow.tracking.fluent: Experiment with name 'foodseg-yolo' does not exist. Creating a new experiment.


New https://pypi.org/project/ultralytics/8.4.90 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.12.8 torch-2.11.0+cpu CPU (AMD Ryzen 7 7730U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=trai

2026/07/08 11:06:45 INFO mlflow.tracking.fluent: Experiment with name '/Shared/Ultralytics' does not exist. Creating a new experiment.


MLflow: logging run_id(6d9c92972fc34ba3982624630da1cc99) to http://mlflowsuperuser:wfu:1H6B_pFLe',2"@cleristonm.duckdns.org:5000
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 768 train, 768 val
Using 0 dataloader workers
Logging results to C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\train
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       1/50         0G          0          0      263.4          0          0          0        768: 100% ━━━━━━━━━━━━ 6/6 14.2s/it 1:25.7ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 5.8s/it 11.6s<36.6s
                   all         17          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, c

c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:839: RuntimeWarning: Mean of empty slice
  i = smooth(f1_curve.mean(0), 0.1).argmax()  # max F1 index
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


...


50 epochs completed in 1.060 hours.
Optimizer stripped from C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\train\weights\last.pt, 20.6MB
Optimizer stripped from C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\train\weights\best.pt, 20.6MB

Validating C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\train\weights\best.pt...
Ultralytics 8.4.46  Python-3.12.8 torch-2.11.0+cpu CPU (AMD Ryzen 7 7730U with Radeon Graphics)
YOLO11s-seg summary (fused): 114 layers, 10,106,677 parameters, 0 gradients, 33.0 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.4s/it 8.7s<26.7s


c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:657: RuntimeWarning: Mean of empty slice
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:703: RuntimeWarning: Mean of empty slice
  y = smooth(py.mean(0), 0.1)
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(

                   all         17          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, cannot compute metrics without labels
Speed: 3.4ms preprocess, 477.2ms inference, 0.0ms loss, 18.7ms postprocess per image
Results saved to C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\train
🏃 View run marvelous-shrimp-762 at: http://mlflowsuperuser:wfu:1H6B_pFLe',2"@cleristonm.duckdns.org:5000/#/experiments/8/runs/6d9c92972fc34ba3982624630da1cc99
🧪 View experiment at: http://mlflowsuperuser:wfu:1H6B_pFLe',2"@cleristonm.duckdns.org:5000/#/experiments/8
MLflow: results logged to http://mlflowsuperuser:wfu:1H6B_pFLe',2"@cleristonm.duckdns.org:5000
MLflow: disable with 'yolo settings mlflow=False'
Ultralytics 8.4.46  Python-3.12.8 torch-2.11.0+cpu CPU (AMD Ryzen 7 7730U with Radeon Graphics)
YOLO11s-seg summary (fused): 114 layers, 10,106,677 parameters,

c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:657: RuntimeWarning: Mean of empty slice
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:703: RuntimeWarning: Mean of empty slice
  y = smooth(py.mean(0), 0.1)
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(

                   all         17          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, cannot compute metrics without labels
Speed: 2.4ms preprocess, 428.5ms inference, 0.0ms loss, 16.5ms postprocess per image
Results saved to C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\val


In [ ]:
metrics = model.val()

new_map = metrics.box.map50 if hasattr(metrics, "box") else metrics.seg.map50

print("New model mAP50:", new_map)



Ultralytics 8.4.46  Python-3.12.8 torch-2.11.0+cpu CPU (AMD Ryzen 7 7730U with Radeon Graphics)
val: Fast image access  (ping: 0.00.0 ms, read: 115.095.4 MB/s, size: 30.2 KB)
val: Scanning C:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\dataset\Images\val.cache... 0 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17  0.0s
WARNING Labels are missing or empty in C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\Images\val.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.9s/it 7.7s<24.2s


c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:657: RuntimeWarning: Mean of empty slice
  ax.plot(px, py.mean(1), linewidth=3, color="blue", label=f"all classes {ap[:, 0].mean():.3f} mAP@0.5")
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\ultralytics\utils\metrics.py:703: RuntimeWarning: Mean of empty slice
  y = smooth(py.mean(0), 0.1)
c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\numpy\_core\_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(

                   all         17          0          0          0          0          0          0          0          0          0
WARNING no labels found in segment set, cannot compute metrics without labels
Speed: 2.4ms preprocess, 423.1ms inference, 0.0ms loss, 15.5ms postprocess per image
Results saved to C:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\runs\segment\val-2
New model mAP50: 0.0


In [ ]:
import json
from pathlib import Path

# ==========================================================
# Dossier du modèle de production
# ==========================================================

CURRENT_MODEL = Path.cwd() / "current-model"
CURRENT_MODEL.mkdir(exist_ok=True)

METRICS_FILE = CURRENT_MODEL / "metrics.json"

# ==========================================================
# Lecture des métriques du modèle courant
# ==========================================================

old_map = 0.0

if METRICS_FILE.exists():
    with open(METRICS_FILE, "r", encoding="utf-8") as f:
        old_map = float(json.load(f).get("mAP50", 0.0))

print(f"Current production mAP50 : {old_map:.4f}")

Current production mAP50 : 0.0000


In [ ]:
from pathlib import Path

PREDICT_SOURCE = DEST / "images" / "val"
PREDICT_OUTPUT = PROJECT_ROOT / "predictions"

results = model.predict(
    source=str(PREDICT_SOURCE),
    conf=0.25,
    save=True,
    project=str(PREDICT_OUTPUT),
    name="validation",
    exist_ok=True,
    max_det=10,
)


image 1/17 c:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\images\val\00004432.jpg: 512x768 (no detections), 471.8ms
image 2/17 c:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\images\val\00004567.jpg: 512x768 (no detections), 315.4ms
image 3/17 c:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\images\val\00004573.jpg: 576x768 (no detections), 447.6ms
image 4/17 c:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\images\val\00004622.jpg: 448x768 (no detections), 349.6ms
image 5/17 c:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\images\val\00004655.jpg: 512x768 (no detections), 263.7ms
image 6/17 c:\Users\cleri\OneDrive - Collge de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHSE\Food Segmentation\dataset\images\

In [ ]:
import json
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd

# =====================================================================
# Définition des chemins
# =====================================================================

RUNS_DIR = PROJECT_ROOT / "runs"

CURRENT_MODEL_DIR = PROJECT_ROOT / "current-model"
CURRENT_MODEL_DIR.mkdir(exist_ok=True)

BEST_MODEL = CURRENT_MODEL_DIR / "best.pt"
METRICS_FILE = CURRENT_MODEL_DIR / "metrics.json"

HISTORY_FILE = PROJECT_ROOT / "all_runs_metrics.csv"

# =====================================================================
# Recherche du dernier entraînement
# =====================================================================

results_files = sorted(
    RUNS_DIR.glob("segment/train*/results.csv"),
    key=lambda p: p.stat().st_mtime,
)

if not results_files:
    raise FileNotFoundError("Aucun fichier results.csv trouvé.")

results_path = results_files[-1]

train_dir = results_path.parent

weights_dir = train_dir / "weights"

best_model_path = weights_dir / "best.pt"

# =====================================================================
# Promotion du modèle
# =====================================================================

promoted = new_map > old_map

if promoted:

    print("New model is better. Promoting.")

    shutil.copy2(best_model_path, BEST_MODEL)

    with open(METRICS_FILE, "w", encoding="utf-8") as f:
        json.dump(
            {"mAP50": float(new_map)},
            f,
            indent=4,
        )

else:

    print("New model is worse. Keeping current production model.")

# =====================================================================
# Lecture des résultats
# =====================================================================

results = pd.read_csv(results_path)

best_row = results.loc[
    results["metrics/mAP50(M)"].idxmax()
]

# =====================================================================
# Résumé de cette exécution
# =====================================================================

run_metrics = pd.DataFrame([{
    "timestamp": datetime.now().isoformat(),
    "seed": SEED,
    "sample_size": len(train_files),
    "val_size": len(val_files),
    "epochs": len(results),
    "best_box_map50": float(best_row["metrics/mAP50(B)"]),
    "best_box_map50_95": float(best_row["metrics/mAP50-95(B)"]),
    "best_mask_map50": float(best_row["metrics/mAP50(M)"]),
    "best_mask_map50_95": float(best_row["metrics/mAP50-95(M)"]),
    "promoted": promoted,
}])

# =====================================================================
# Historique des exécutions
# =====================================================================

if HISTORY_FILE.exists():
    history = pd.read_csv(HISTORY_FILE)
    history = pd.concat(
        [history, run_metrics],
        ignore_index=True,
    )
else:
    history = run_metrics

history.to_csv(HISTORY_FILE, index=False)

print("\nHistorique sauvegardé.")
print(history.tail())

# =====================================================================
# Arrêt si le modèle n'est pas meilleur
# =====================================================================

if not promoted:
    raise SystemExit()

New model is worse. Keeping current production model.

Historique sauvegardé.
                    timestamp    seed  sample_size  val_size  epochs  \
0  2026-07-08T12:11:13.347742  750339           42        17      50   

   best_box_map50  best_box_map50_95  best_mask_map50  best_mask_map50_95  \
0             0.0                0.0              0.0                 0.0   

   promoted  
0     False  


SystemExit: 

c:\Users\cleri\OneDrive - Collège de Bois-de-Boulogne\curso_ai\PROJET DE SYNTHÈSE\Food Segmentation\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
